# TrustSkin AI — Phase 5: Controlled Ablation Ladder, Multi-Seed Robustness & Deep Ensemble

**CS320 Fall 2026 — Project P08: Reliable Recognition of Difficult Skin Lesions**  
**Repository:** https://github.com/SikandarHussain6858/TrustSkinAI

This notebook runs the controlled Phase 5 experiments on a Google Colab GPU (~1.5–2.5 hours total on T4):
1. **Ablation R1:** Pretrained ResNet-18, plain CE, NO augmentation (`configs/abl_r1_resnet_ce_noaug.yaml`)
2. **Ablation R2:** R1 + dermoscopy augmentation (`configs/abl_r2_resnet_ce_aug.yaml`)
3. **Ablation R3s:** R2 + square-root dampened class weights (`configs/abl_r3s_resnet_sqrtw_aug.yaml`)
4. **3-Seed Robustness + Deep Ensemble:** Trains seeds `43` and `44` of the best R3/R3s config (chosen on **validation** performance!) and compares **Deterministic Softmax vs. MC Dropout vs. Deep Ensemble**.
5. **Statistical & Difficult-Case Suite:** Re-runs `05_compare_experiments.py` (ablation ladder) and `07_difficult_case_eval.py` (difficult subsets, bootstrap 95% CIs, paired McNemar tests).

In [1]:
# Cell 1: Verify Colab GPU runtime
import torch
!nvidia-smi
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

Thu Nov 26 05:50:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2: Clone/pull TrustSkinAI, set PROJECT_ROOT & PYTHONPATH=src
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/SikandarHussain6858/TrustSkinAI.git'
cwd = Path.cwd().resolve()
if (cwd / 'pyproject.toml').exists() and (cwd / 'configs' / 'base.yaml').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'pyproject.toml').exists() and (cwd.parent / 'configs' / 'base.yaml').exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path('/content/TrustSkinAI')
    if not PROJECT_ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull'], check=True)

os.chdir(PROJECT_ROOT)
src_dir = str((PROJECT_ROOT / 'src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
os.environ['PYTHONPATH'] = src_dir
print('PROJECT_ROOT:', Path.cwd())

PROJECT_ROOT: /content/TrustSkinAI


In [3]:
# Cell 3: Fast multi-stream download of MedIMeta derm.zip from Zenodo if not already present
import json
import shutil
import subprocess
import urllib.request
import zipfile
from pathlib import Path

data_root = Path('data')
derm_dir = data_root / 'derm'
zip_path = data_root / 'derm.zip'

if (derm_dir / 'info.yaml').exists():
    print('Dataset already ready at:', derm_dir.resolve())
else:
    if derm_dir.exists():
        shutil.rmtree(derm_dir)
    data_root.mkdir(parents=True, exist_ok=True)
    if shutil.which('aria2c') is None:
        !apt-get update -qq && apt-get install -y -qq aria2
    with urllib.request.urlopen('https://zenodo.org/api/records/7884735') as resp:
        record = json.loads(resp.read().decode('utf-8'))
    zip_file = next(f for f in record['files'] if f['key'].lower() == 'derm.zip')
    download_url = zip_file['links']['self']
    subprocess.run([
        'aria2c', '-x', '16', '-s', '16', '-k', '1M', '--continue=true',
        '--summary-interval=10', '-d', str(data_root), '-o', 'derm.zip', download_url
    ], check=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        top_names = {name.split('/')[0] for name in zf.namelist() if name}
    target = data_root if 'derm' in top_names else derm_dir
    target.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', str(zip_path), '-d', str(target)], check=True)
    zip_path.unlink(missing_ok=True)
    print('Dataset extracted cleanly!')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libcares2:amd64.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../libcares2_1.27.0-1.0ubuntu1.1_amd64.deb ...
Unpacking libcares2:amd64 (1.27.0-1.0ubuntu1.1) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.37.0+debian-1build3_amd64.deb ...
Unpacking libaria2-0:amd64 (1.37.0+debian-1build3) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.37.0+debian-1build3_amd64.deb ...
Unpacking aria2 (1.37.0+debian-1build3) ...
Setting up libcares2:amd64 (1.27.0-1.0ubuntu1.1) ...
Setting up libaria2-0:amd64 (1.37.0+debian-1build3) ...
Setting up aria2 (1.37.0+debian-1build3) ...
Processing triggers for man-db (2.12.0-4build2) ...
Processing tri

In [4]:
!git -C /content/TrustSkinAI pull
!ls /content/TrustSkinAI/configs

Already up to date.
abl_r1_resnet_ce_noaug.yaml    baseline_cnn.yaml     resnet18_mc.yaml
abl_r2_resnet_ce_aug.yaml      base.yaml
abl_r3s_resnet_sqrtw_aug.yaml  improved_mc_cnn.yaml


In [5]:
# Cell 4: Verify single-factor config diffs (R1 -> R2 -> R3s) & run pytest
from trustskin.utils import load_config

r1 = load_config('configs/abl_r1_resnet_ce_noaug.yaml')
r2 = load_config('configs/abl_r2_resnet_ce_aug.yaml')
r3s = load_config('configs/abl_r3s_resnet_sqrtw_aug.yaml')
print('R1 -> R2 diff: dataset.augment =', r1['dataset']['augment'], '->', r2['dataset']['augment'])
print('R2 -> R3s diff: training.loss =', r2['training']['loss'], '->', r3s['training']['loss'], '(power =', r3s['training'].get('class_weight_power'), ')')

!pytest tests/ -v

R1 -> R2 diff: dataset.augment = False -> True
R2 -> R3s diff: training.loss = cross_entropy -> weighted_ce (power = 0.5 )
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/TrustSkinAI
configfile: pyproject.toml
plugins: langsmith-0.14.17, anyio-4.16.0, typeguard-4.6.0, platformdirs-4.14.3
collected 17 items                                                             

tests/test_calibration.py::test_ece_and_brier_perfect_predictions PASSED [  5%]
tests/test_calibration.py::test_temperature_scaler_improves_overconfident_nll PASSED [ 11%]
tests/test_datamodule_local.py::test_local_medimeta_derm_splits_and_shapes PASSED [ 17%]
tests/test_models.py::test_baseline_cnn_forward_shape PASSED             [ 23%]
tests/test_models.py::test_uncertainty_cnn_mc_dropout_stochasticity PASSED [ 29%]
tests/test_models.py::test_configs_load_and_build_m

In [6]:
# Cell 5: Ablation R1 — Pretrained ResNet-18, plain CE, NO augmentation
!python scripts/03_train_improved.py --config configs/abl_r1_resnet_ce_noaug.yaml
!python scripts/04_calibrate_and_select.py \
    --config configs/abl_r1_resnet_ce_noaug.yaml \
    --checkpoint checkpoints/abl_r1_resnet_ce_noaug_best.pt

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 251MB/s]
Epoch 01/20 | Train Loss: 0.6246, F1: 0.5593 | Val Loss: 0.5999, Acc: 0.7876, Macro-F1: 0.6581
Epoch 02/20 | Train Loss: 0.3646, F1: 0.7687 | Val Loss: 0.4658, Acc: 0.8394, Macro-F1: 0.6826
Epoch 03/20 | Train Loss: 0.2416, F1: 0.8544 | Val Loss: 0.4241, Acc: 0.8653, Macro-F1: 0.8270
Epoch 04/20 | Train Loss: 0.1280, F1: 0.9347 | Val Loss: 0.5929, Acc: 0.7772, Macro-F1: 0.5898
Epoch 05/20 | Train Loss: 0.0888, F1: 0.9491 | Val Loss: 0.6074, Acc: 0.8497, Macro-F1: 0.7251
Epoch 06/20 | Train Loss: 0.0528, F1: 0.9753 | Val Loss: 0.5000, Acc: 0.8394, Macro-F1: 0.7117
Epoch 07/20 | Train Loss: 0.0506, F1: 0.9739 | Val Loss: 0.6525, Acc: 0.8238, Macro-F1: 0.6177
Epoch 08/20 | Train Loss: 0.0192, F1: 0.9914 | Val Loss: 0.5248, Acc: 0.8860, Macro-F1: 0.8009
Epoch 09/20 | Train Loss: 0.0130, F1: 0.9948 | Val Loss: 0.5390, Ac

In [7]:
# Cell 6: Ablation R2 — R1 + Dermoscopy Data Augmentation (plain CE)
!python scripts/03_train_improved.py --config configs/abl_r2_resnet_ce_aug.yaml
!python scripts/04_calibrate_and_select.py \
    --config configs/abl_r2_resnet_ce_aug.yaml \
    --checkpoint checkpoints/abl_r2_resnet_ce_aug_best.pt

Epoch 01/20 | Train Loss: 0.6763, F1: 0.4937 | Val Loss: 0.6197, Acc: 0.7876, Macro-F1: 0.5030
Epoch 02/20 | Train Loss: 0.5203, F1: 0.6243 | Val Loss: 0.5739, Acc: 0.7772, Macro-F1: 0.6803
Epoch 03/20 | Train Loss: 0.4614, F1: 0.6898 | Val Loss: 0.5742, Acc: 0.7772, Macro-F1: 0.6310
Epoch 04/20 | Train Loss: 0.4091, F1: 0.7373 | Val Loss: 0.4857, Acc: 0.8135, Macro-F1: 0.6634
Epoch 05/20 | Train Loss: 0.3780, F1: 0.7467 | Val Loss: 0.4855, Acc: 0.8135, Macro-F1: 0.6514
Epoch 06/20 | Train Loss: 0.3431, F1: 0.7927 | Val Loss: 0.5364, Acc: 0.8135, Macro-F1: 0.7653
Epoch 07/20 | Train Loss: 0.3220, F1: 0.8075 | Val Loss: 0.4928, Acc: 0.8342, Macro-F1: 0.7238
Epoch 08/20 | Train Loss: 0.2819, F1: 0.8325 | Val Loss: 0.5646, Acc: 0.8135, Macro-F1: 0.7162
Epoch 09/20 | Train Loss: 0.2489, F1: 0.8594 | Val Loss: 0.4594, Acc: 0.8549, Macro-F1: 0.6776
Epoch 10/20 | Train Loss: 0.2058, F1: 0.8942 | Val Loss: 0.4796, Acc: 0.8342, Macro-F1: 0.7752
Epoch 11/20 | Train Loss: 0.1820, F1: 0.9030 | Val

In [8]:
# Cell 7: Ablation R3s — R2 + Square-Root Dampened Class Weights (power = 0.5)
!python scripts/03_train_improved.py --config configs/abl_r3s_resnet_sqrtw_aug.yaml
!python scripts/04_calibrate_and_select.py \
    --config configs/abl_r3s_resnet_sqrtw_aug.yaml \
    --checkpoint checkpoints/abl_r3s_resnet_sqrtw_aug_best.pt

[abl_r3s_resnet_sqrtw_aug] Using class weights (power=0.5): [0.601 0.245 0.885 1.109 0.605 1.871 1.684]
Epoch 01/20 | Train Loss: 0.9671, F1: 0.5091 | Val Loss: 0.7154, Acc: 0.7876, Macro-F1: 0.5498
Epoch 02/20 | Train Loss: 0.7353, F1: 0.6279 | Val Loss: 0.9305, Acc: 0.7513, Macro-F1: 0.4751
Epoch 03/20 | Train Loss: 0.6573, F1: 0.6796 | Val Loss: 1.1672, Acc: 0.6321, Macro-F1: 0.4926
Epoch 04/20 | Train Loss: 0.5847, F1: 0.7117 | Val Loss: 0.6529, Acc: 0.7927, Macro-F1: 0.7299
Epoch 05/20 | Train Loss: 0.5311, F1: 0.7474 | Val Loss: 0.5460, Acc: 0.8394, Macro-F1: 0.6936
Epoch 06/20 | Train Loss: 0.4620, F1: 0.7869 | Val Loss: 0.8247, Acc: 0.7772, Macro-F1: 0.6286
Epoch 07/20 | Train Loss: 0.4407, F1: 0.7828 | Val Loss: 0.6393, Acc: 0.8290, Macro-F1: 0.6578
Epoch 08/20 | Train Loss: 0.3819, F1: 0.8231 | Val Loss: 0.4912, Acc: 0.8549, Macro-F1: 0.7564
Epoch 09/20 | Train Loss: 0.3386, F1: 0.8476 | Val Loss: 0.6087, Acc: 0.8394, Macro-F1: 0.6530
Epoch 10/20 | Train Loss: 0.2823, F1: 0.8

In [9]:
# Cell 8: Multi-Seed Robustness (Seeds 43 & 44) for R3s -> Forms 3-Model Deep Ensemble
# (If validation selects linear R3 instead of R3s, switch BEST_CFG to configs/resnet18_mc.yaml)
BEST_CFG = 'configs/abl_r3s_resnet_sqrtw_aug.yaml'
SEED0_CKPT = 'checkpoints/abl_r3s_resnet_sqrtw_aug_best.pt'

!python scripts/03_train_improved.py --config {BEST_CFG} --seed 43 --exp-name r3s_seed43
!python scripts/04_calibrate_and_select.py --config {BEST_CFG} --checkpoint checkpoints/r3s_seed43_best.pt --exp-name r3s_seed43

!python scripts/03_train_improved.py --config {BEST_CFG} --seed 44 --exp-name r3s_seed44
!python scripts/04_calibrate_and_select.py --config {BEST_CFG} --checkpoint checkpoints/r3s_seed44_best.pt --exp-name r3s_seed44

[r3s_seed43] Using class weights (power=0.5): [0.601 0.245 0.885 1.109 0.605 1.871 1.684]
Epoch 01/20 | Train Loss: 1.0073, F1: 0.4924 | Val Loss: 0.9520, Acc: 0.7098, Macro-F1: 0.4953
Epoch 02/20 | Train Loss: 0.7521, F1: 0.6220 | Val Loss: 0.6235, Acc: 0.7876, Macro-F1: 0.6327
Epoch 03/20 | Train Loss: 0.6432, F1: 0.6933 | Val Loss: 0.6720, Acc: 0.7772, Macro-F1: 0.6200
Epoch 04/20 | Train Loss: 0.6248, F1: 0.6891 | Val Loss: 0.6602, Acc: 0.8187, Macro-F1: 0.6889
Epoch 05/20 | Train Loss: 0.5086, F1: 0.7561 | Val Loss: 0.6087, Acc: 0.8653, Macro-F1: 0.7520
Epoch 06/20 | Train Loss: 0.4890, F1: 0.7572 | Val Loss: 0.6076, Acc: 0.7979, Macro-F1: 0.6870
Epoch 07/20 | Train Loss: 0.4181, F1: 0.8182 | Val Loss: 0.7959, Acc: 0.7772, Macro-F1: 0.6503
Epoch 08/20 | Train Loss: 0.4008, F1: 0.8080 | Val Loss: 0.6108, Acc: 0.8446, Macro-F1: 0.7946
Epoch 09/20 | Train Loss: 0.3491, F1: 0.8375 | Val Loss: 0.8053, Acc: 0.8135, Macro-F1: 0.6629
Epoch 10/20 | Train Loss: 0.2963, F1: 0.8657 | Val Loss

In [10]:
# Cell 9: Uncertainty Method Comparison (Deterministic vs. MC Dropout vs. 3-Seed Deep Ensemble)
!python scripts/08_ensemble_eval.py \
    --config {BEST_CFG} \
    --checkpoints {SEED0_CKPT} checkpoints/r3s_seed43_best.pt checkpoints/r3s_seed44_best.pt \
    --mc-samples 25

=== Uncertainty Method Comparison ===
                             method  accuracy  balanced_accuracy  macro_f1    ece  brier  aurc_conf  aurc_ent  error_auroc_conf  error_auroc_ent  error_auroc_mi
Deterministic Softmax (1 model, TS)    0.7884             0.7046    0.7058 0.0505 0.2980     0.0629    0.0627            0.8388           0.8393          0.5000
     MC Dropout (1 model, T=25, TS)    0.7917             0.7098    0.7104 0.0450 0.2970     0.0626    0.0620            0.8342           0.8369          0.8043
        Deep Ensemble (3 seeds, TS)    0.7970             0.7570    0.7458 0.0204 0.2815     0.0573    0.0575            0.8450           0.8441          0.7484


In [11]:
# Cell 10: Compile Ablation Ladder, Difficult-Case Subsets, Bootstrap 95% CIs, McNemar Tests & Package Zip
!python scripts/05_compare_experiments.py --output-dir outputs
!python scripts/07_difficult_case_eval.py --output-dir outputs

!zip -rq outputs/trustskin_phase5_results.zip checkpoints outputs/tables outputs/figures outputs/logs
from pathlib import Path
import pandas as pd
bundle = Path('outputs/trustskin_phase5_results.zip')
print('Packaged Phase 5 bundle:', bundle.resolve(), f'({bundle.stat().st_size / 1e6:.2f} MB)')
for tbl in ['ablation_ladder.csv', 'uncertainty_method_comparison.csv', 'difficult_case_metrics.csv', 'mcnemar_tests.csv']:
    p = Path('outputs/tables') / tbl
    if p.exists():
        print('\n===', tbl, '===')
        display(pd.read_csv(p))

Saved comparison table to outputs/tables/experiment_comparison.csv:
              experiment  accuracy  balanced_accuracy  macro_f1  ece_before  ece_after  brier_after   aurc  error_auroc_ent  temperature
  abl_r1_resnet_ce_noaug    0.7639             0.5377    0.5804      0.0952     0.0779       0.3418 0.0747           0.8359       1.0882
    abl_r2_resnet_ce_aug    0.7983             0.6729    0.6790      0.0926     0.0726       0.2955 0.0578           0.8389       1.1627
abl_r3s_resnet_sqrtw_aug    0.7884             0.7037    0.7072      0.0693     0.0486       0.2971 0.0626           0.8407       1.1405
              r3s_seed43    0.7758             0.7385    0.7169      0.0918     0.0589       0.3057 0.0623           0.8609       1.2377
              r3s_seed44    0.7149             0.6823    0.6631      0.1156     0.0904       0.3813 0.1002           0.8217       1.1245

Saved ordered ablation ladder to outputs/tables/ablation_ladder.csv
Saved combined Risk-Coverage plot to outp

,experiment,accuracy,balanced_accuracy,macro_f1,ece_before,ece_after,brier_after,aurc,error_auroc_ent,temperature,delta_bal_acc,delta_macro_f1,delta_aurc
0,abl_r1_resnet_ce_noaug,0.7639,0.5377,0.5804,0.0952,0.0779,0.3418,0.0747,0.8359,1.0882,0.0000,0.0000,0.0000
1,abl_r2_resnet_ce_aug,0.7983,0.6729,0.6790,0.0926,0.0726,0.2955,0.0578,0.8389,1.1627,0.1352,0.0986,-0.0169
2,abl_r3s_resnet_sqrtw_aug,0.7884,0.7037,0.7072,0.0693,0.0486,0.2971,0.0626,0.8407,1.1405,0.0308,0.0282,0.0048



=== uncertainty_method_comparison.csv ===


,method,accuracy,balanced_accuracy,macro_f1,ece,brier,aurc_conf,aurc_ent,error_auroc_conf,error_auroc_ent,error_auroc_mi
0,"Deterministic Softmax (1 model, TS)",0.7884,0.7046,0.7058,0.0505,0.2980,0.0629,0.0627,0.8388,0.8393,0.5000
1,"MC Dropout (1 model, T=25, TS)",0.7917,0.7098,0.7104,0.0450,0.2970,0.0626,0.0620,0.8342,0.8369,0.8043
2,"Deep Ensemble (3 seeds, TS)",0.7970,0.7570,0.7458,0.0204,0.2815,0.0573,0.0575,0.8450,0.8441,0.7484



=== difficult_case_metrics.csv ===


,model,subset,n,accuracy,balanced_accuracy,macro_f1,mel_recall,aurc
0,abl_r1_resnet_ce_noaug,all_test,1512,0.7639,0.5377,0.5804,0.5029,0.0747
1,abl_r1_resnet_ce_noaug,hard_6_non_nv,603,0.4959,0.4703,0.4775,0.5029,0.4278
2,abl_r1_resnet_ce_noaug,confusable_triad_mel_nv_bkl,1297,0.8134,0.6551,0.3043,0.5029,0.0521
3,abl_r2_resnet_ce_aug,all_test,1512,0.7983,0.6729,0.6790,0.4035,0.0578
4,abl_r2_resnet_ce_aug,hard_6_non_nv,603,0.5904,0.6291,0.5753,0.4035,0.2560
5,abl_r2_resnet_ce_aug,confusable_triad_mel_nv_bkl,1297,0.8211,0.6693,0.3081,0.4035,0.0483
6,abl_r3s_resnet_sqrtw_aug,all_test,1512,0.7884,0.7037,0.7072,0.7076,0.0626
7,abl_r3s_resnet_sqrtw_aug,hard_6_non_nv,603,0.6667,0.6761,0.6168,0.7076,0.1621
8,abl_r3s_resnet_sqrtw_aug,confusable_triad_mel_nv_bkl,1297,0.7995,0.7191,0.3111,0.7076,0.0591
9,deep_ensemble,all_test,1512,0.7970,0.7570,0.7458,0.7661,0.0573



=== mcnemar_tests.csv ===


EmptyDataError: No columns to parse from file

In [12]:
!mkdir -p /content/drive/MyDrive/TrustSkinAI/phase5_results
!cp /content/TrustSkinAI/outputs/trustskin_phase5_results.zip /content/drive/MyDrive/TrustSkinAI/phase5_results/
!ls -lh /content/drive/MyDrive/TrustSkinAI/phase5_results/

total 202M
-rw-r--r-- 1 root root 202M Nov 26 06:42 trustskin_phase5_results.zip


In [14]:
import shutil
from google.colab import drive

# 1. Remove the dummy local /content/drive folder so Drive can mount cleanly
shutil.rmtree('/content/drive', ignore_errors=True)

# 2. Mount your real Google Drive
drive.mount('/content/drive', force_remount=True)

# 3. Copy Phase 5 zip into its own folder on your real Drive
!mkdir -p /content/drive/MyDrive/TrustSkinAI/phase5_results
!cp /content/TrustSkinAI/outputs/trustskin_phase5_results.zip /content/drive/MyDrive/TrustSkinAI/phase5_results/
!ls -lh /content/drive/MyDrive/TrustSkinAI/phase5_results/

# 4. Flush upload immediately
drive.flush_and_unmount()
print("Done! Refresh MyDrive/TrustSkinAI/phase5_results/ in Google Drive.")

Mounted at /content/drive
total 202M
-rw------- 1 root root 202M Nov 26 06:49 trustskin_phase5_results.zip
Done! Refresh MyDrive/TrustSkinAI/phase5_results/ in Google Drive.
